# Import Libraries

In [1]:
import pyspark
import pandas as pd
import numpy as np
import math

In [ ]:
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField, StructType, StringType, LongType, IntegerType, FloatType
from pyspark.sql.functions import col, column
from pyspark.sql.functions import expr
from pyspark.sql.functions import split
from pyspark.sql import Row
from pyspark.sql import functions as F
from functools import reduce
from operator import add
from pyspark.ml.feature import VectorAssembler


# Create SparkSessions and SparkContext

In [3]:
ss=SparkSession.builder.master("local").appName("Spotify Playlist Generator").getOrCreate()

Error: LinkageError occurred while loading main class org.apache.spark.launcher.Main
	java.lang.UnsupportedClassVersionError: org/apache/spark/launcher/Main has been compiled by a more recent version of the Java Runtime (class file version 61.0), this version of the Java Runtime only recognizes class file versions up to 55.0
/Users/praptikanani/Library/Python/3.9/lib/python/site-packages/pyspark/bin/spark-class: line 97: CMD: bad array subscript
head: illegal line count -- -1


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
ss.sparkContext.setCheckpointDir("~/scratch")

# Read Data

In [ ]:
spotify_DF = ss.read.csv("./spotify_dataset.csv", header=True, inferSchema=True)

In [ ]:
spotify_DF.printSchema()

In [ ]:
spotify_DF.first()
# Expected CSV columns (minimum): title, artist, album, release_year, explicit, playlist_title
expected_cols = ['song', 'Artist(s)','Energy','Genre', 'Album', 'Tempo', 'Loudness', 'Popularity', 'Liveness', 'Acousticness', 'Speechiness', 'Danceability', 'Positiveness', 'Time signature', 'Instrumentalness', 'Release Date','Explicit']
for col in expected_cols:
    if col not in spotify_DF.columns:
        raise ValueError(f"Missing expected column: {col}")

# Clean & Normalize text columns 

In [ ]:
def norm(col):
    return F.lower(F.trim(F.regexp_replace(col, r"\s+", " ")))

df = (
    spotify_DF.withColumn("song_norm", norm(F.col("song")))
      .withColumn("artist_norm", norm(F.col("`Artist(s)`")))
      .withColumn("genre_norm", norm(F.col("Genre")))
      .withColumn("album_norm", norm(F.col("Album")))
)
df.select("song","song_norm","Artist(s)","artist_norm","Genre","genre_norm").show(10, truncate=False)

# Add a unique ID & drop duplicates

# Handle nulls & fix data types

# Keyword flags + coarse genre

# Proxy features (0–1) from genre + keywords

# Hard labels (priority)

In [ ]:
POP_IS_0_100 = True   # if popularity already 0-1, set False
POP_HIGH  = 70.0 if POP_IS_0_100 else 0.70
POP_MED   = 50.0 if POP_IS_0_100 else 0.50

hard = (
  # --- Era / keyword buckets first (deterministic) ---
  F.when(any_rx(txt, r"(christmas|xmas|holiday|noel)"),                      F.lit("Christmas"))
   .when(any_rx(txt, r"(disney|pixar)"),                                     F.lit("Childhood Disney Music"))
   .when(any_rx(txt, r"(original broadway cast|musical|motion picture soundtrack|cast|ensemble)"),
                                                                             F.lit("Musicals"))
   .when((between("release_year", 2010, 2019)) & any_rx("genre_norm", r"(pop|pop[- ]?punk|rock|hip ?hop|r&b)"),
                                                                             F.lit("2010s Pop Hits"))
   .when(between("release_year", 2000, 2009),                                F.lit("2000s Throwbacks"))
   .when((between("release_year", 1990, 1999)) & any_rx("genre_norm", r"(alt|grunge|rock)"),
                                                                             F.lit("90s Alt Rock"))
   .when(between("release_year", 1969, 1989),                                F.lit("Vintage Classics (70-80s)"))

  # Sad girl autumn
   .when(
        (low("Energy") | med("Energy")) &
        between("Tempo", 60, 100) &
        (low("Loudness") | med("Loudness")) &
        none_rx("genre_norm", r"(pop|hip ?hop|rap|edm|house|fast)") &
        any_rx("genre_norm", r"(indie|alt|folk)") &
        low("Liveness") &
        high("Acousticness") &
        low("Speechiness") &
        low("Danceability") &
        low("Positiveness") &
        (F.col("Explicit") == F.lit(False)),
        F.lit("Sad Girl Autumn")
   )

  # Study
   .when(
        (low("Energy") | med("Energy")) &
        between("Tempo", 50, 90) &
        low("Loudness") &
        none_rx("genre_norm", r"(pop|hip ?hop|rap|edm|house|fast)") &
        any_rx("genre_norm", r"(indie|alt ?pop|lo[- ]?fi|jazz)") &
        low("Liveness") &
        high("Acousticness") &
        low("Speechiness") &
        low("Danceability") &
        low("Positiveness") &
        (F.col("Explicit") == F.lit(False)),
        F.lit("Study")
   )

  # Christmas 
   .when(
        (med("Energy") | high("Energy")) &
        between("Tempo", 60, 130) &
        (med("Loudness") | high("Loudness")) &
        (F.col("Popularity") >= POP_HIGH) &
        high("Liveness") &
        high("Acousticness") &
        med("Speechiness") &
        (low("Danceability") | med("Danceability")) &
        high("Positiveness") &
        (F.col("`Time signature`") == 3) &  # 3/4
        high("Instrumentalness") &
        (F.col("Explicit") == F.lit(False)),
        F.lit("Christmas")
   )

  # Road trip
   .when(
        (med("Energy") | high("Energy")) &
        between("Tempo", 60, 100) &
        (med("Loudness") | high("Loudness")) &
        (med("Popularity") | (F.col("Popularity") >= POP_HIGH)) &
        (med("Liveness") | high("Liveness")) &
        med("Danceability") &
        (med("Positiveness") | high("Positiveness")) &
        high("Instrumentalness") &
        (F.col("Explicit") == F.lit(False)),
        F.lit("Road Trip")
   )

  # Driving
   .when(
        (low("Energy") | med("Energy")) &
        between("Tempo", 60, 80) &
        (low("Loudness") | med("Loudness")) &
        low("Liveness") &
        high("Acousticness") &
        low("Speechiness") &
        low("Danceability") &
        low("Positiveness") &
        high("Instrumentalness") &
        (F.col("Explicit") == F.lit(False)),
        F.lit("Driving")
   )

  # Musicals 
   .when(
        (low("Energy") | med("Energy")) &
        (low("Loudness") | med("Loudness") | high("Loudness")) &
        (F.col("Popularity") >= POP_HIGH) &
        (med("Liveness") | high("Liveness")) &
        high("Acousticness") &
        high("Speechiness") &
        (low("Danceability") | med("Danceability") | high("Danceability")) &
        (low("Positiveness") | med("Positiveness") | high("Positiveness")) &
        (F.col("Explicit") == F.lit(False)),
        F.lit("Musicals")
   )

  # 2010s Pop Hits
   .when(
        (med("Energy") | high("Energy")) &
        between("Tempo", 80, 130) &
        (med("Loudness") | high("Loudness")) &
        (F.col("Popularity") >= POP_HIGH) &
        (med("Liveness") | high("Liveness")),
        F.lit("2010s Pop Hits")
   )

  # 90s Alt Rock 
   .when(
        high("Energy") &
        between("Tempo", 110, 140) &
        high("Loudness") &
        (F.col("Popularity") >= POP_HIGH),
        F.lit("90s Alt Rock")
   )

  # Vintage classics 
   .when(
        med("Energy") &
        between("Tempo", 60, 130) &
        (med("Loudness") | high("Loudness")) &
        (F.col("Popularity") >= POP_HIGH) &
        (med("Liveness") | high("Liveness")) &
        high("Acousticness") &
        high("Speechiness") &
        high("Danceability") &
        high("Positiveness"),
        F.lit("Vintage Classics (70-80s)")
   )

  # Childhood Disney Music
   .when(
        between("Tempo", 60, 120) &
        (F.col("Popularity") >= POP_HIGH) &
        (med("Liveness") | high("Liveness")) &
        high("Acousticness") &
        high("Speechiness") &
        high("Danceability") &
        high("Positiveness") &
        high("Instrumentalness") &
        (F.col("Explicit") == F.lit(False)),
        F.lit("Childhood Disney Music")
   )

  # Summer beach day
   .when(
        high("Energy") &
        between("Tempo", 100, 130) &
        high("Loudness") &
        high("Danceability") &
        high("Positiveness") &
        (F.col("Popularity") >= POP_HIGH),
        F.lit("Summer Beach Day")
   )

  # Autumn cozy
   .when(
        (low("Energy") | med("Energy")) &
        between("Tempo", 60, 80) &
        med("Loudness") &
        (low("Danceability") | med("Danceability")) &
        (F.col("Popularity") >= POP_MED),
        F.lit("Autumn Cozy")
   )

  # Campfire nights
   .when(
        med("Energy") &
        between("Tempo", 110, 130) &
        (med("Loudness") | high("Loudness")) &
        (med("Popularity") | (F.col("Popularity") >= POP_HIGH)) &
        (med("Liveness") | high("Liveness")) &
        high("Acousticness") &
        high("Speechiness") &
        med("Danceability") &
        high("Positiveness") &
        (F.col("Explicit") == F.lit(False)),
        F.lit("Campfire Nights")
   )

  # Indie chill
   .when(
        low("Energy") &
        between("Tempo", 70, 100) &
        any_rx("genre_norm", r"(indie|alternative)") &
        (low("Liveness") | med("Liveness")) &
        high("Acousticness") &
        low("Speechiness") &
        low("Danceability") &
        (med("Positiveness") | (F.col("Positiveness") == 0.5)) &
        high("Instrumentalness"),
        F.lit("Indie Chill")
   )

  # City night stroll
   .when(
        low("Energy") &
        between("Tempo", 70, 100) &
        any_rx("genre_norm", r"(ambient|classical|jazz)") &
        (low("Liveness") | med("Liveness")) &
        high("Acousticness") &
        low("Speechiness") &
        low("Danceability") &
        (med("Positiveness") | (F.col("Positiveness") == 0.5)) &
        high("Instrumentalness"),
        F.lit("City Night Stroll")
   )

  # Villain arc gym music
   .when(
        high("Energy") &
        between("Tempo", 110, 150) &
        high("Loudness") &
        (med("Popularity") | (F.col("Popularity") >= POP_HIGH)) &
        (med("Liveness") | high("Liveness")) &
        high("Acousticness") &   # you listed high
        high("Speechiness") &
        high("Danceability") &
        high("Positiveness"),
        F.lit("Villain Arc Gym Music")
   )

  # I’m unbothered era (sassy)
   .when(
        high("Energy") &
        between("Tempo", 100, 130) &
        high("Loudness") &
        (med("Popularity") | (F.col("Popularity") >= POP_HIGH)) &
        (med("Liveness") | high("Liveness")) &
        high("Acousticness") &
        high("Speechiness") &
        high("Danceability") &
        high("Positiveness"),
        F.lit("I’m Unbothered Era (Sassy)")
   )

  # Slow morning
   .when(
        (low("Energy") | med("Energy")) &
        between("Tempo", 60, 80) &
        (low("Loudness") | med("Loudness")) &
        (low("Liveness") | med("Liveness")) &
        high("Acousticness") &
        (low("Speechiness") | med("Speechiness")) &
        low("Danceability") &
        (low("Positiveness") | med("Positiveness")) &
        (F.col("`Time signature`") == 3),
        F.lit("Slow Morning")
   )

  # Post breakup depressed
   .when(
        low("Energy") &
        between("Tempo", 60, 100) &
        (low("Loudness") | med("Loudness")),
        F.lit("Post Breakup Depressed")
   )

  # Midnight existential
   .when(
        (low("Energy") | med("Energy")) &
        between("Tempo", 60, 80) &
        (low("Loudness") | med("Loudness")) &
        high("Liveness") &
        high("Speechiness") &
        high("Danceability") &
        high("Positiveness") &
        high("Instrumentalness"),
        F.lit("Midnight Existential")
   )

  # Hot girl walk
   .when(
        between("Tempo", 60, 130) &
        med("Loudness") &
        (med("Popularity") | (F.col("Popularity") >= POP_HIGH)) &
        high("Liveness") &
        high("Acousticness") &
        (F.col("Speechiness") >= 0.5) &   # "medium"
        (F.col("Danceability") >= 0.5) &  # "medium"
        high("Positiveness") &
        (F.col("`Time signature`") == 3) &  # 3/4
        (F.col("Instrumentalness") >= 0.5),
        F.lit("Hot Girl Walk")
   )

  # Main character energy
   .when(
        high("Energy") &
        between("Tempo", 70, 100) &
        low("Loudness") &
        (med("Popularity") | (F.col("Popularity") >= POP_HIGH)) &
        (med("Liveness") | high("Liveness")) &
        high("Acousticness") &
        high("Speechiness") &
        high("Danceability") &
        high("Positiveness"),
        F.lit("Main Character Energy")
   )

  # Dream / escapist
   .when(
        low("Energy") &
        between("Tempo", 100, 130) &
        high("Loudness") &
        any_rx("genre_norm", r"(experimental|psychedelic|psychedelic rock|dream ?pop)") &
        (F.col("Popularity") >= POP_MED) &
        (low("Liveness") | med("Liveness")) &
        high("Acousticness") &
        low("Speechiness") &
        low("Danceability") &
        low("Positiveness") &
        high("Instrumentalness"),
        F.lit("Dream/Escapist")
   )

  # Going out
   .when(
        (med("Energy") | high("Energy")) &
        between("Tempo", 80, 130) &
        high("Loudness") &
        (F.col("Popularity") >= POP_MED) &
        high("Liveness") &
        (F.col("Acousticness") >= 0.5) &   # "medium"
        (F.col("Speechiness") >= 0.5) &    # "medium-high"
        high("Danceability") &
        high("Positiveness") &
        (F.col("`Time signature`") == 4) &  # 4/4
        (F.col("Instrumentalness") <= 0.4),
        F.lit("Going Out")
   )
)

# Soft label (simple scorer UDF, pick max)

In [ ]:
df = (df
 .withColumn("energy",       bump("energy",       0.70, gb=="edm"))
 .withColumn("danceability", bump("danceability", 0.70, gb=="edm"))
 .withColumn("loudness",     bump("loudness",     0.70, gb=="edm"))
 .withColumn("tempo",        bump("tempo",        0.65, gb=="edm"))

 # rnb/hiphop: +0.10 energy, +0.10 dance, +0.15 speech, +0.05 loud
 .withColumn("energy",       bump("energy",       0.10, gb=="hiphop"))
 .withColumn("danceability", bump("danceability", 0.10, gb=="hiphop"))
 .withColumn("speechiness",  bump("speechiness",  0.15, gb=="hiphop"))
 .withColumn("loudness",     bump("loudness",     0.05, gb=="hiphop"))
 # rock: +0.40 energy, +0.40 loud, +0.35 tempo, +0.20 dance
 .withColumn("energy",       bump("energy",       0.40, gb=="rock"))
 .withColumn("loudness",     bump("loudness",     0.40, gb=="rock"))
 .withColumn("tempo",        bump("tempo",        0.35, gb=="rock"))
 .withColumn("danceability", bump("danceability", 0.20, gb=="rock"))

 # indie/folk: −0.10 energy, +0.40 acoustic, −0.05 valence
 .withColumn("energy",       bump("energy",      -0.10, gb=="indie_folk"))
 .withColumn("acousticness", bump("acousticness", 0.40, gb=="indie_folk"))
 .withColumn("valence",      bump("valence",     -0.05, gb=="indie_folk"))

# lofi/jazz: −0.30 energy, +0.10 dance, +0.20 acoustic, +0.05 live
 .withColumn("energy",       bump("energy",      -0.30, gb=="lofi_jazz"))
 .withColumn("danceability", bump("danceability", 0.10, gb=="lofi_jazz"))
 .withColumn("acousticness", bump("acousticness", 0.20, gb=="lofi_jazz"))
 .withColumn("liveness",     bump("liveness",     0.05, gb=="lofi_jazz"))

 # soundtrack: +0.20 live, +0.10 speech, −0.05 energy
 .withColumn("liveness",     bump("liveness",     0.20, gb=="soundtrack"))
 .withColumn("speechiness",  bump("speechiness",  0.10, gb=="soundtrack"))
 .withColumn("energy",       bump("energy",      -0.05, gb=="soundtrack"))

 # disney/holiday: +0.20 valence, +0.10 live
 .withColumn("valence",      bump("valence",      0.20, gb=="disney_holiday"))
 .withColumn("liveness",     bump("liveness",     0.10, gb=="disney_holiday"))
# pop: +0.30 energy, +0.30 dance, +0.10 loud, +0.30 tempo
 .withColumn("energy",       bump("energy",       0.30, gb=="pop"))
 .withColumn("danceability", bump("danceability", 0.30, gb=="pop"))
 .withColumn("loudness",     bump("loudness",     0.10, gb=="pop"))
 .withColumn("tempo",        bump("tempo",        0.30, gb=="pop"))
 # classical: +0.10 energy, +0.10 dance, +0.10 loud, +0.20 tempo
 .withColumn("energy",       bump("energy",       0.10, gb=="classical"))
 .withColumn("danceability", bump("danceability", 0.10, gb=="classical"))
 .withColumn("loudness",     bump("loudness",     0.10, gb=="classical"))
 .withColumn("tempo",        bump("tempo",        0.20, gb=="classical"))

 # latin/reggae: +0.40 energy, +0.50 dance, +0.30 loud, +0.50 tempo
 .withColumn("energy",       bump("energy",       0.40, gb=="latin_reggae"))
 .withColumn("danceability", bump("danceability", 0.50, gb=="latin_reggae"))
 .withColumn("loudness",     bump("loudness",     0.30, gb=="latin_reggae"))
 .withColumn("tempo",        bump("tempo",        0.50, gb=="latin_reggae"))

# spiritual/worship: -0.10 energy, +0.10 dance, +0.10 loud, +0.10 tempo, +0.20 acoustic
 .withColumn("energy",       bump("energy",      -0.10, gb=="spiritual"))
 .withColumn("danceability", bump("danceability", 0.10, gb=="spiritual"))
 .withColumn("loudness",     bump("loudness",     0.10, gb=="spiritual"))
 .withColumn("tempo",        bump("tempo",        0.10, gb=="spiritual"))
 .withColumn("acousticness", bump("acousticness", 0.20, gb=="spiritual"))

 # jazz/soul: -0.10 energy, +0.10 dance, +0.10 loud, +0.10 tempo, +0.40 acoustic
 .withColumn("energy",       bump("energy",      -0.10, gb=="jazz_soul"))
 .withColumn("danceability", bump("danceability", 0.10, gb=="jazz_soul"))
 .withColumn("loudness",     bump("loudness",     0.10, gb=="jazz_soul"))
 .withColumn("tempo",        bump("tempo",        0.10, gb=="jazz_soul"))
 .withColumn("acousticness", bump("acousticness", 0.40, gb=="jazz_soul"))
)

# Assign categories to existing songs based on label rules

In [ ]:
df = df.withColumn(
    "assigned_category",
    F.when(F.col("hard_label").isNotNull(), F.col("hard_label"))
     .when(F.col("soft_score") >= F.lit(0.25), F.col("soft_label"))
)
df_agg = (
    df.groupBy("assigned_category")
      .agg(F.count("*").alias("song_count"))
)
total = df.count()
df_agg = df_agg.withColumn("percent", (F.col("song_count") / total) * 100)

df_agg.orderBy(F.desc("song_count")).show(50, truncate=False)

In [ ]:
feature_cols = [
    "Energy", "Tempo", "Loudness", "Danceability", "Positiveness",
    "Acousticness", "Speechiness", "Liveness", "Instrumentalness"
]

cat_avgs = ( df.groupBy("assigned_category")
      .agg(*[F.avg(c).alias(c) for c in feature_cols])
)

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_vectorize = assembler.transform(cat_avgs)
df_vectorize.show(truncate=False)

a = df_vectorize.select("assigned_category", "features").alias("a")
b = df_vectorize.select("assigned_category", "features").alias("b")
pairs = (
    a.crossJoin(b)
     .select(
        F.col("a.assigned_category").alias("cat_i"),
        F.col("b.assigned_category").alias("cat_j"),
        F.col("a.features").alias("vector1"),
        F.col("b.features").alias("vector2"),
     )
)

def dot(v1, v2):
    return float(v1.dot(v2))

def norm(vec): 
    return float(vec.norm(2))

dot_udf = F.udf(dot,F.DoubleType())
norm_udf = F.udf(norm, F.DoubleType())

cosine_similarity = (
    pairs
      .withColumn("dot_product", dot_udf(F.col("vector1"), F.col("vector2")))
      .withColumn("norm_1",      norm_udf(F.col("vector1")))
      .withColumn("norm_2",      norm_udf(F.col("vector2")))
      .withColumn("cosine_similarity",
                  F.col("dot_product") / (F.col("norm_1") * F.col("norm_2")))
)
cosine_similarity.show()



In [ ]:
ss.stop()